# Self Concept Literature - Network Analysis

This notebook builds a questionnaire validity/correlation network from `tier1_validity_extraction.xlsx`.

The central rule is strict: the graph node universe is defined only by questionnaires/scales that appear in `focal_questionnaire`. Rows where `related_questionnaire` points to outside instruments are excluded from the graph and saved for QA.


## 1. Load the Workbook and Validate Columns

The workbook is loaded with pandas. Column names are normalized only for minor spelling/case/spacing differences. Major missing columns stop the notebook with a clear error.


In [ ]:
from pathlib import Path
from collections import Counter
import html
import math
import re
import textwrap

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from pyvis.network import Network

GRAPH_TITLE = "Self concept literature - network analysis"
INPUT_FILENAME = "tier1_validity_extraction.xlsx"
PREFERRED_SHEET = "validity_edges"

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / INPUT_FILENAME
OUTPUT_FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

INTERACTIVE_GRAPH_PATH = OUTPUT_FIGURES_DIR / "self_concept_network.html"
STATIC_GRAPH_PATH = OUTPUT_FIGURES_DIR / "self_concept_network.png"
CLEANED_EDGES_PATH = OUTPUT_TABLES_DIR / "cleaned_network_edges.csv"
NODE_LIST_PATH = OUTPUT_TABLES_DIR / "network_nodes.csv"
EXCLUDED_ROWS_QA_PATH = OUTPUT_TABLES_DIR / "excluded_rows_QA.csv"
EXCLUDED_NON_FOCAL_PATH = OUTPUT_TABLES_DIR / "excluded_non_focal_related_questionnaires.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Expected input file not found: {DATA_PATH}")

xlsx = pd.ExcelFile(DATA_PATH)
SHEET_NAME = PREFERRED_SHEET if PREFERRED_SHEET in xlsx.sheet_names else xlsx.sheet_names[0]
df_raw = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)

print(f"Workbook: {DATA_PATH}")
print(f"Available sheets: {xlsx.sheet_names}")
print(f"Selected sheet: {SHEET_NAME}")
print("Actual column names:")
for i, column in enumerate(df_raw.columns, start=1):
    print(f"  {i}. {column!r}")

REQUIRED_COLUMNS = [
    "focal_questionnaire",
    "related_questionnaire",
    "relation_value",
    "significance",
    "relation_source_type",
]

def canonicalize_column_name(column):
    """Normalize column names for minor case/spacing/punctuation differences."""
    text = str(column).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return re.sub(r"_+", "_", text).strip("_")

actual_by_canonical = {}
duplicate_canonical_names = []
for column in df_raw.columns:
    canonical = canonicalize_column_name(column)
    if canonical in actual_by_canonical:
        duplicate_canonical_names.append((canonical, actual_by_canonical[canonical], column))
    else:
        actual_by_canonical[canonical] = column

if duplicate_canonical_names:
    print("Warning: duplicate columns after normalization:")
    for canonical, first, second in duplicate_canonical_names:
        print(f"  {canonical!r}: {first!r} and {second!r}")

missing_columns = [column for column in REQUIRED_COLUMNS if canonicalize_column_name(column) not in actual_by_canonical]
if missing_columns:
    raise ValueError(
        "Missing required columns after minor normalization: "
        + ", ".join(missing_columns)
        + "\nActual columns were: "
        + ", ".join(map(str, df_raw.columns))
    )

column_mapping = {
    actual_by_canonical[canonicalize_column_name(expected)]: expected
    for expected in REQUIRED_COLUMNS
}

df = df_raw.rename(columns=column_mapping).copy()

print("\nColumn mapping used:")
for actual, expected in column_mapping.items():
    print(f"  {actual!r} -> {expected!r}")


## 2. Clean Questionnaire Names and Parse Relation Values

Questionnaire labels are cleaned for matching while preserving the readable display text. Relation values are parsed from common forms such as `.45`, `-.32`, `r = .45`, and `r=.45`.


In [ ]:
def clean_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).strip())


def questionnaire_key(value):
    """Stable matching key for questionnaire names."""
    return clean_text(value).casefold()


def parse_relation_value(value):
    """Parse a numeric correlation-like value from common validity/correlation text."""
    if pd.isna(value):
        return pd.NA
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return float(value)

    text = clean_text(value)
    if not text:
        return pd.NA

    text = (
        text.replace("−", "-")
        .replace("–", "-")
        .replace("—", "-")
        .replace(",", ".")
    )

    labelled_match = re.search(
        r"(?:^|[\s,;(])(?:r|sp|sr|semipartial|semi-partial|partial)\s*=?\s*([-+]?(?:\d*\.\d+|\d+\.?\d*))",
        text,
        flags=re.IGNORECASE,
    )
    if labelled_match:
        candidate = labelled_match.group(1)
    else:
        fallback_match = re.search(r"[-+]?(?:\d*\.\d+|\d+\.?\d*)", text)
        if not fallback_match:
            return pd.NA
        candidate = fallback_match.group(0)

    try:
        return float(candidate)
    except ValueError:
        return pd.NA


def split_questionnaire_label(label):
    """Split labels like '(SCC): Self-Concept Clarity Scale' into acronym and name."""
    text = clean_text(label).replace("*", "")
    match = re.match(r"^(\([^:]+\))\s*:\s*(.+)$", text)
    if match:
        return match.group(1).strip(), match.group(2).strip()
    match = re.match(r"^([^:]{2,40})\s*:\s*(.+)$", text)
    if match:
        return match.group(1).strip(), match.group(2).strip()
    return "", text


def format_relation_number(value):
    text = f"{float(value):.2f}"
    if text.startswith("-0"):
        return "-" + text[2:]
    if text.startswith("0"):
        return text[1:]
    return text


def format_edge_label(value, source_type):
    source_type = clean_text(source_type).casefold()
    number = format_relation_number(value)
    if source_type == "regression_semipartial correlation":
        return f"sp={number}"
    suffix = "*" if source_type == "convergent_validity" else ""
    return f"r={number}{suffix}"


def edge_color(value):
    if float(value) > 0:
        return "#d73027"  # red for positive values
    if float(value) < 0:
        return "#2c7fb8"  # blue for negative values
    return "#666666"


def show_table(table, title=None):
    if title:
        print(title)
    try:
        display(table)
    except NameError:
        print(table.to_string(index=False))


df["focal_clean"] = df["focal_questionnaire"].map(clean_text)
df["related_clean"] = df["related_questionnaire"].map(clean_text)
df["focal_key"] = df["focal_clean"].map(questionnaire_key)
df["related_key"] = df["related_clean"].map(questionnaire_key)
df["significance_clean"] = df["significance"].map(lambda value: clean_text(value).casefold())
df["source_type_clean"] = df["relation_source_type"].map(lambda value: clean_text(value).casefold())
df["source_type_clean"] = df["source_type_clean"].str.replace(r"\s+", " ", regex=True)
df["relation_value_numeric"] = pd.to_numeric(df["relation_value"].map(parse_relation_value), errors="coerce")


## 3. Define Nodes, Filter Edges, and Build QA Tables

The focal set is built only from `focal_questionnaire`. Every focal questionnaire becomes a node. Edges are included only when the related questionnaire is also focal and all significance/source/value filters pass.


In [ ]:
ALLOWED_SOURCE_TYPES = {
    "correlation_only",
    "convergent_validity",
    "regression_semipartial correlation",
}

focal_display_by_key = (
    df.loc[df["focal_key"] != "", ["focal_key", "focal_clean"]]
    .drop_duplicates("focal_key")
    .set_index("focal_key")["focal_clean"]
    .to_dict()
)
focal_keys = set(focal_display_by_key)

nodes_df = pd.DataFrame(
    [
        {"questionnaire_key": key, "questionnaire_label": label}
        for key, label in sorted(focal_display_by_key.items(), key=lambda item: item[1].casefold())
    ]
)
if not nodes_df.empty:
    split_labels = nodes_df["questionnaire_label"].map(split_questionnaire_label)
    nodes_df["label_acronym"] = split_labels.map(lambda item: item[0])
    nodes_df["label_full_name"] = split_labels.map(lambda item: item[1])
else:
    nodes_df["label_acronym"] = []
    nodes_df["label_full_name"] = []

df["has_focal_questionnaire"] = df["focal_key"] != ""
df["has_related_questionnaire"] = df["related_key"] != ""
df["is_significant"] = df["significance_clean"] == "yes"
df["source_type_allowed"] = df["source_type_clean"].isin(ALLOWED_SOURCE_TYPES)
df["relation_value_parseable"] = df["relation_value_numeric"].notna()
df["focal_in_focal_set"] = df["focal_key"].isin(focal_keys)
df["related_in_focal_set"] = df["related_key"].isin(focal_keys)

include_mask = (
    df["has_focal_questionnaire"]
    & df["has_related_questionnaire"]
    & df["is_significant"]
    & df["source_type_allowed"]
    & df["relation_value_parseable"]
    & df["focal_in_focal_set"]
    & df["related_in_focal_set"]
)


def exclusion_reasons(row):
    reasons = []
    if not row["has_focal_questionnaire"]:
        reasons.append("missing_focal_questionnaire")
    if not row["has_related_questionnaire"]:
        reasons.append("missing_related_questionnaire")
    if not row["is_significant"]:
        reasons.append("not_significant")
    if not row["source_type_allowed"]:
        reasons.append("source_type_not_allowed")
    if not row["relation_value_parseable"]:
        reasons.append("relation_value_unparseable")
    if not row["focal_in_focal_set"]:
        reasons.append("focal_not_in_focal_set")
    if row["has_related_questionnaire"] and not row["related_in_focal_set"]:
        reasons.append("related_not_in_focal_set")
    return "; ".join(reasons)


eligible_edges = df.loc[include_mask].copy()
excluded_rows = df.loc[~include_mask].copy()
excluded_rows["exclusion_reason"] = excluded_rows.apply(exclusion_reasons, axis=1)

excluded_non_focal_related = df.loc[
    df["has_related_questionnaire"] & ~df["related_in_focal_set"]
].copy()
excluded_non_focal_related["exclusion_reason"] = "related_not_in_focal_set"

eligible_edges["focal_label"] = eligible_edges["focal_key"].map(focal_display_by_key)
eligible_edges["related_label"] = eligible_edges["related_key"].map(focal_display_by_key)
eligible_edges["edge_label"] = eligible_edges.apply(
    lambda row: format_edge_label(row["relation_value_numeric"], row["source_type_clean"]), axis=1
)
eligible_edges["edge_color"] = eligible_edges["relation_value_numeric"].map(edge_color)
eligible_edges["edge_abs_relation"] = eligible_edges["relation_value_numeric"].abs()

node_u = []
node_v = []
for _, row in eligible_edges.iterrows():
    first, second = sorted([row["focal_key"], row["related_key"]])
    node_u.append(first)
    node_v.append(second)
eligible_edges["node_u_key"] = node_u
eligible_edges["node_v_key"] = node_v
eligible_edges["node_u_label"] = eligible_edges["node_u_key"].map(focal_display_by_key)
eligible_edges["node_v_label"] = eligible_edges["node_v_key"].map(focal_display_by_key)

# Build graph-level edges. If repeated rows exist for the same pair, the visual edge uses the mean value,
# while cleaned_network_edges.csv still preserves the row-level eligible edge list.
graph_edge_records = []
if not eligible_edges.empty:
    for (source_key, target_key), group in eligible_edges.groupby(["node_u_key", "node_v_key"], sort=False):
        mean_value = group["relation_value_numeric"].mean()
        source_types = sorted(group["source_type_clean"].dropna().unique())
        if len(group) <= 3:
            label = "; ".join(group["edge_label"].tolist())
        else:
            label = f"{format_edge_label(mean_value, group.iloc[0]['source_type_clean'])} mean; n={len(group)}"
        graph_edge_records.append(
            {
                "source_key": source_key,
                "target_key": target_key,
                "source_label": focal_display_by_key[source_key],
                "target_label": focal_display_by_key[target_key],
                "relation_value": mean_value,
                "abs_relation_value": abs(mean_value),
                "edge_label": label,
                "edge_color": edge_color(mean_value),
                "n_relations": len(group),
                "relation_source_type": "; ".join(source_types),
            }
        )

graph_edges = pd.DataFrame(graph_edge_records)

connected_keys = set()
if not graph_edges.empty:
    connected_keys.update(graph_edges["source_key"])
    connected_keys.update(graph_edges["target_key"])
nodes_df["is_connected"] = nodes_df["questionnaire_key"].isin(connected_keys)
nodes_df["degree"] = nodes_df["questionnaire_key"].map(
    lambda key: int((eligible_edges["node_u_key"].eq(key) | eligible_edges["node_v_key"].eq(key)).sum())
    if not eligible_edges.empty else 0
)

# QA prints requested near the beginning of the notebook.
print("Number of total rows loaded:", len(df))
print("Number of unique focal questionnaires:", len(focal_keys))
print("Number of rows with significant values:", int(df["is_significant"].sum()))
print("Number of rows with allowed relation_source_type:", int(df["source_type_allowed"].sum()))
print("Number of rows where related_questionnaire is also in the focal questionnaire set:", int(df["related_in_focal_set"].sum()))
print("Final number of edges included:", len(eligible_edges))

reason_counter = Counter()
for reasons in excluded_rows["exclusion_reason"]:
    for reason in str(reasons).split("; "):
        if reason:
            reason_counter[reason] += 1
exclusion_counts_df = pd.DataFrame(
    sorted(reason_counter.items()), columns=["exclusion_reason", "row_count"]
)
show_table(exclusion_counts_df, "\nNumber of excluded rows by exclusion reason:")

summary_rows = [
    {"metric": "total focal nodes", "value": len(nodes_df)},
    {"metric": "connected focal nodes", "value": int(nodes_df["is_connected"].sum())},
    {"metric": "isolated focal nodes", "value": int((~nodes_df["is_connected"]).sum())},
    {"metric": "positive edges", "value": int((eligible_edges["relation_value_numeric"] > 0).sum())},
    {"metric": "negative edges", "value": int((eligible_edges["relation_value_numeric"] < 0).sum())},
]
source_type_counts = (
    eligible_edges["source_type_clean"]
    .value_counts()
    .rename_axis("relation_source_type")
    .reset_index(name="edge_count")
)
summary_table = pd.concat(
    [
        pd.DataFrame(summary_rows),
        source_type_counts.rename(columns={"relation_source_type": "metric", "edge_count": "value"}),
    ],
    ignore_index=True,
)
show_table(summary_table, "\nSummary table:")


## 4. Save Cleaned Tables

The edge table keeps eligible row-level relationships. The graph may aggregate duplicate pair relationships visually, but the CSV preserves the cleaned source rows.


In [ ]:
preferred_edge_columns = [
    "focal_questionnaire",
    "related_questionnaire",
    "relation_value",
    "relation_value_numeric",
    "significance",
    "relation_source_type",
    "source_type_clean",
    "edge_label",
    "focal_label",
    "related_label",
    "node_u_label",
    "node_v_label",
    "edge_abs_relation",
]
optional_edge_columns = [column for column in ["article_reference", "notes"] if column in eligible_edges.columns]
edge_columns = [column for column in preferred_edge_columns + optional_edge_columns if column in eligible_edges.columns]

eligible_edges[edge_columns].to_csv(CLEANED_EDGES_PATH, index=False)
nodes_df.to_csv(NODE_LIST_PATH, index=False)
excluded_rows.to_csv(EXCLUDED_ROWS_QA_PATH, index=False)
excluded_non_focal_related.to_csv(EXCLUDED_NON_FOCAL_PATH, index=False)

print(f"Saved cleaned edge list: {CLEANED_EDGES_PATH}")
print(f"Saved node list: {NODE_LIST_PATH}")
print(f"Saved excluded rows QA table: {EXCLUDED_ROWS_QA_PATH}")
print(f"Saved non-focal related questionnaire QA table: {EXCLUDED_NON_FOCAL_PATH}")


## 5. Build the NetworkX Graph

All focal questionnaires are added as nodes, including isolated nodes. Eligible focal-to-focal relationships become edges.


In [ ]:
G = nx.Graph()
for _, row in nodes_df.iterrows():
    G.add_node(
        row["questionnaire_key"],
        label=row["questionnaire_label"],
        acronym=row["label_acronym"],
        full_name=row["label_full_name"],
        degree=int(row["degree"]),
        is_connected=bool(row["is_connected"]),
    )

for _, row in graph_edges.iterrows():
    G.add_edge(
        row["source_key"],
        row["target_key"],
        relation_value=float(row["relation_value"]),
        abs_relation_value=float(row["abs_relation_value"]),
        label=row["edge_label"],
        color=row["edge_color"],
        n_relations=int(row["n_relations"]),
        relation_source_type=row["relation_source_type"],
    )

print(f"Graph nodes: {G.number_of_nodes()}")
print(f"Graph edges: {G.number_of_edges()}")


## 6. Static PNG Graph

Positive edges are red, negative edges are blue, and edge width scales with the absolute relation value. Edge labels use the same color as their edge.


In [ ]:
def static_node_label(label):
    acronym, full_name = split_questionnaire_label(label)
    if acronym and full_name:
        return f"{acronym}\n{textwrap.fill(full_name, width=24)}"
    return textwrap.fill(label, width=26)

n_nodes = max(G.number_of_nodes(), 1)
if G.number_of_edges() > 0:
    layout_k = max(0.9, 3.8 / math.sqrt(n_nodes))
    pos = nx.spring_layout(G, seed=42, k=layout_k, iterations=600, scale=5.5, weight=None)
else:
    pos = nx.circular_layout(G, scale=5.5)

fig_width = max(13, min(22, 0.85 * n_nodes))
fig_height = max(10, fig_width * 0.75)
fig, ax = plt.subplots(figsize=(fig_width, fig_height))
ax.set_title(GRAPH_TITLE, fontsize=18, pad=20)
ax.axis("off")

node_colors = ["#f5f5f5" if G.nodes[node]["is_connected"] else "#ffffff" for node in G.nodes]
node_edge_colors = ["#555555" if G.nodes[node]["is_connected"] else "#bbbbbb" for node in G.nodes]
node_sizes = [700 + 120 * G.degree(node) for node in G.nodes]

nx.draw_networkx_nodes(
    G,
    pos,
    node_size=node_sizes,
    node_color=node_colors,
    edgecolors=node_edge_colors,
    linewidths=1.2,
    ax=ax,
)

if G.number_of_edges() > 0:
    edge_colors = [data["color"] for _, _, data in G.edges(data=True)]
    edge_widths = [1.2 + 7.5 * data["abs_relation_value"] for _, _, data in G.edges(data=True)]
    nx.draw_networkx_edges(
        G,
        pos,
        edge_color=edge_colors,
        width=edge_widths,
        alpha=0.78,
        ax=ax,
    )

node_labels = {node: static_node_label(data["label"]) for node, data in G.nodes(data=True)}
nx.draw_networkx_labels(
    G,
    pos,
    labels=node_labels,
    font_size=8,
    font_color="#222222",
    font_weight="bold",
    ax=ax,
)

# Draw edge labels manually so each label can match its edge color.
for source, target, data in G.edges(data=True):
    x = (pos[source][0] + pos[target][0]) / 2
    y = (pos[source][1] + pos[target][1]) / 2
    ax.text(
        x,
        y,
        data["label"],
        color=data["color"],
        fontsize=8,
        fontweight="bold",
        ha="center",
        va="center",
        bbox={"boxstyle": "round,pad=0.2", "facecolor": "white", "edgecolor": "none", "alpha": 0.72},
    )

fig.tight_layout()
fig.savefig(STATIC_GRAPH_PATH, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved static graph: {STATIC_GRAPH_PATH}")


## 7. Interactive HTML Graph

The interactive graph uses PyVis. The acronym part of each questionnaire label is marked as bold with larger font settings where the browser's vis-network label renderer supports HTML multi-font labels. Full questionnaire names also appear in hover tooltips.


In [ ]:
def interactive_node_label(label):
    acronym, full_name = split_questionnaire_label(label)
    if acronym and full_name:
        return f"<b>{html.escape(acronym)}</b>\n{html.escape(textwrap.fill(full_name, width=34))}"
    return html.escape(textwrap.fill(label, width=34))


def interactive_node_title(label, degree):
    acronym, full_name = split_questionnaire_label(label)
    if acronym and full_name:
        title = f"<b>{html.escape(acronym)}</b><br>{html.escape(full_name)}"
    else:
        title = html.escape(label)
    return f"{title}<br><br>Eligible focal-network degree: {degree}"

net = Network(
    height="850px",
    width="100%",
    bgcolor="#ffffff",
    font_color="#222222",
    notebook=False,
    cdn_resources="remote",
)
net.barnes_hut(
    gravity=-50000,
    central_gravity=0.18,
    spring_length=230,
    spring_strength=0.035,
    damping=0.09,
    overlap=0.7,
)

for node, data in G.nodes(data=True):
    degree = int(data["degree"])
    net.add_node(
        node,
        label=interactive_node_label(data["label"]),
        title=interactive_node_title(data["label"], degree),
        shape="dot",
        size=16 + 3 * min(degree, 8),
        color={
            "background": "#f5f5f5" if data["is_connected"] else "#ffffff",
            "border": "#555555" if data["is_connected"] else "#bbbbbb",
            "highlight": {"background": "#fff7bc", "border": "#111111"},
        },
        font={
            "multi": "html",
            "size": 16,
            "face": "arial",
            "bold": {"size": 23, "face": "arial", "vadjust": -2},
        },
    )

for source, target, data in G.edges(data=True):
    relation_value = data["relation_value"]
    net.add_edge(
        source,
        target,
        label=data["label"],
        title=(
            f"{html.escape(data['label'])}<br>"
            f"Relation value: {relation_value:.3f}<br>"
            f"Source type: {html.escape(data['relation_source_type'])}<br>"
            f"Rows represented: {data['n_relations']}"
        ),
        color=data["color"],
        width=1.2 + 7.5 * data["abs_relation_value"],
        value=max(1, 10 * data["abs_relation_value"]),
        font={"color": data["color"], "size": 16, "strokeWidth": 4, "strokeColor": "#ffffff"},
        smooth={"enabled": True, "type": "dynamic"},
    )

net.set_options(
    """
var options = {
  "nodes": {
    "borderWidth": 1,
    "font": {"multi": "html"}
  },
  "edges": {
    "selectionWidth": 2,
    "font": {"align": "middle"},
    "smooth": {"enabled": true, "type": "dynamic"}
  },
  "interaction": {
    "hover": true,
    "tooltipDelay": 120,
    "navigationButtons": true,
    "keyboard": true
  },
  "physics": {
    "solver": "barnesHut",
    "barnesHut": {
      "gravitationalConstant": -50000,
      "centralGravity": 0.18,
      "springLength": 230,
      "springConstant": 0.035,
      "damping": 0.09,
      "avoidOverlap": 0.7
    },
    "stabilization": {"enabled": true, "iterations": 300}
  }
}
"""
)

net.write_html(str(INTERACTIVE_GRAPH_PATH), notebook=False)

# Add a visible title to the generated HTML page.
html_text = INTERACTIVE_GRAPH_PATH.read_text(encoding="utf-8")
title_markup = (
    f"<h2 style='font-family: Arial, sans-serif; margin: 18px 18px 4px 18px;'>"
    f"{html.escape(GRAPH_TITLE)}</h2>"
)
if GRAPH_TITLE not in html_text:
    html_text = html_text.replace("<body>", f"<body>\n{title_markup}", 1)
    INTERACTIVE_GRAPH_PATH.write_text(html_text, encoding="utf-8")

print(f"Saved interactive graph: {INTERACTIVE_GRAPH_PATH}")


## 8. Output Review

The files below are the expected reproducible outputs from the notebook.


In [ ]:
for path in [
    INTERACTIVE_GRAPH_PATH,
    STATIC_GRAPH_PATH,
    CLEANED_EDGES_PATH,
    NODE_LIST_PATH,
    EXCLUDED_ROWS_QA_PATH,
    EXCLUDED_NON_FOCAL_PATH,
]:
    print(path, "exists=", path.exists())
